# 02 · Data sources, audit and alignment (CP1)

This notebook **calls** the CP1 rules; they live in `src/pavana_ml/data/research.py` so they can
be tested without a notebook and without network access.

Experiment target (`ml/PLAN.md`): the **CAMS PM2.5 model field on one grid cell**, not a station
observation. Nothing here is station validation or proof of operational forecast skill.

Outputs:

    ml/data/interim/hourly_aligned.parquet
    ml/data/interim/audit_report.json

Not done here: calling a provider, reading a credential, filling a missing value, or treating a
hotspot detection as a confirmed fire.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pavana_ml.data.research import (
    DATASET_REVISION,
    MANIFEST_PATHS,
    download_snapshot,
    prepare_cached_data,
    save_prepared,
)
from pavana_ml.paths import DATA_DIR

pd.set_option("display.width", 140)
print("data dir:", DATA_DIR)

## 1 · Snapshot

Experiments read the dataset at a **pinned commit**, never `main`, so a later upload cannot change
what a result was computed from. `download_snapshot` resolves the revision once, reads the file
list at that commit, and verifies every file against the manifest's SHA256 before use.

Data is already cached locally, so nothing is fetched unless `DOWNLOAD_IF_MISSING` is set.

Live provider loaders exist and their parsers are unit-tested (`firms.tidy_hotspots`,
`firms.combine_hotspots`, `open_meteo.parse_hourly`); the HTTP calls (`firms.fetch_range`,
`open_meteo.fetch_weather_hourly`, `open_meteo.fetch_air_quality_hourly`) need network and a
FIRMS key, and are only used to refill the cache. `openaq.fetch_station_pm25` is still
unimplemented — and it is not on this path: OpenAQ has no station inside the Kalimantan bbox.
This notebook reads the published snapshot from disk and calls no provider.


In [ ]:
DOWNLOAD_IF_MISSING = False  # set True on a fresh clone, or in Colab once DATA_DIR is writable


def cached(relative: str) -> bool:
    """Present at its Hub path, or at the data root (the local BMKG layout)."""
    return (DATA_DIR / relative).exists() or (DATA_DIR / Path(relative).name).exists()


missing = sorted(path for path in MANIFEST_PATHS.values() if not cached(path))
if missing and DOWNLOAD_IF_MISSING:
    download_snapshot(DATA_DIR, DATASET_REVISION)
    missing = sorted(path for path in MANIFEST_PATHS.values() if not cached(path))

print("pinned revision :", DATASET_REVISION)
print("missing locally :", missing or "none")

## 2 · Validate and align

`prepare_cached_data` reads only local files and refuses to continue when:

- a required column is absent, or a provenance sidecar says the wrong `kind`
  (`reanalysis` for ERA5, `model` for CAMS);
- `valid_at` is naive, duplicated, or not on the hour;
- a value is **physically impossible** (negative concentration or wind speed, humidity outside
  0–100 %, pressure outside 300–1100 hPa). Extreme but possible values are counted in the report
  and **kept** — nothing is trimmed, winsorised or interpolated;
- the two sources did not return exactly one grid cell each, or a hotspot key repeats.

Both sources are sorted and reindexed to a complete hourly index *before* any shift or rolling, so
a missing hour becomes `NaN` instead of silently joining two distant hours.

In [ ]:
hourly, hotspots, report = prepare_cached_data(DATA_DIR)
print(f"aligned hours : {len(hourly):,}")
print(f"hotspot rows  : {len(hotspots):,}")
print(f"period (UTC)  : {report['aligned']['start_utc']} -> {report['aligned']['end_utc']}")
hourly.head(3)

**Sanity check.** 2024-01-01 00:00 → 2026-10-14 23:00 UTC is 24 432 whole hours. This snapshot must
keep all of them: dropping an hour would hide a gap, adding one would invent a timestamp.

In [ ]:
EXPECTED_HOURS = 24_432
assert report["aligned"]["rows"] == EXPECTED_HOURS, report["aligned"]["rows"]
assert report["aligned"]["unique_timestamps"]
assert report["aligned"]["hourly_interval"]
pd.DataFrame(report["hour_regularity"]).T

## 3 · Provenance and integrity

`revision_verified` is only `True` when the manifest bytes and every consumed file hash agree. If
they do not, `revision` stays `null` — a commit is never attached to bytes that did not match it.

In [ ]:
snapshot = report["snapshot"]
print("repo            :", snapshot["repo_id"])
print("revision        :", snapshot["revision"], "| verified:", snapshot["revision_verified"])
print("evidence        :", snapshot["revision_source"] or snapshot["reason"])
print("manifest sha256 :", snapshot["manifest_sha256"])
pd.DataFrame(snapshot["files"]).T

In [ ]:
sources = pd.DataFrame(report["sources"]).T
sources[["kind", "provider", "rows", "start_utc", "end_utc"]]

In [ ]:
context = sources[sources["kind"] == "context"]
context.reindex(columns=["rows", "group_column", "group_count", "start_utc", "end_utc", "note"])

The requested point and the cell that answered are **not the same coordinate**. Keep the returned
cell: at ~0.1° it identifies which series was actually served, and two neighbouring cells can
differ by tens of µg/m³.

In [ ]:
requested = {name: report["sources"][name]["requested_point"] for name in ("weather", "cams")}
pd.concat([pd.DataFrame(report["grid_cells"]).T, pd.DataFrame(requested).T], axis=1)

## 4 · Missingness

A missing value stays `NaN`. `null` is never coerced to `0`, and no column is interpolated.

In [ ]:
missing = pd.Series(report["missing_by_column"], name="missing_rows").sort_values(ascending=False)
print("aligned rows:", report["aligned"]["rows"])
missing[missing > 0]

## 5 · EDA — CAMS PM2.5 model field

Values are the CAMS global model field (~45 km, 3-hourly, interpolated to hourly by Open-Meteo)
for one grid cell. They are not measurements from a station.

In [ ]:
pd.Series(report["cams_pm25_quantiles"], name="cams_pm25_ug_m3")

In [ ]:
monthly = pd.DataFrame(report["cams_pm25_monthly"]).T
monthly.index.name = "month_utc"
monthly

In [ ]:
series = hourly.set_index("valid_at")["cams_pm25_ug_m3"]
boundary = report["distribution_shift_probe"]["sensor_mix_boundary_utc"][:7]

fig, axes = plt.subplots(3, 1, figsize=(11, 9))
series.resample("D").mean().plot(ax=axes[0], lw=0.8, color="steelblue")
axes[0].set_title("CAMS PM2.5 daily mean (model field, one grid cell)")
axes[0].set_ylabel("ug/m3")

monthly["mean"].astype(float).plot(kind="bar", ax=axes[1], color="darkorange")
if boundary in monthly.index:
    axes[1].axvline(monthly.index.get_loc(boundary) - 0.5, color="crimson", ls="--")
axes[1].set_title(f"Monthly mean; dashed line = FIRMS sensor-mix boundary ({boundary})")
axes[1].set_ylabel("ug/m3")

series.plot(kind="hist", bins=60, ax=axes[2], color="seagreen")
axes[2].set_title("Distribution of hourly CAMS PM2.5")
axes[2].set_xlabel("ug/m3")
fig.tight_layout()

**Distribution shift, handled as a probe only.** The level changes sharply across the FIRMS
sensor-mix boundary — but the detector/processing mix changes on the same date, so the shift cannot
be attributed to air quality. It is not evidence of a smoke episode.

In [ ]:
probe = report["distribution_shift_probe"]
print("boundary     :", probe["sensor_mix_boundary_utc"])
print("mean before  :", probe["before_mean_ug_m3"])
print("mean after   :", probe["after_mean_ug_m3"])
print("caveat       :", probe["caveat"])

## 6 · EDA — hotspot detections

Counts are **records in this cache**. A detection is not a confirmed fire, and a month with few or
zero detections is not evidence that the satellite covered it or that nothing burned. Per-sensor
collection completeness is unverified, and the NRT windows keep growing as they fill in.

In [ ]:
coverage = pd.DataFrame(report["hotspot_audit"]["by_source_sensor"]).T
coverage

In [ ]:
counts = pd.Series(report["hotspot_audit"]["monthly_counts"], name="records")
counts.index.name = "month_utc"
ax = counts.plot(kind="bar", figsize=(11, 3.5), color="firebrick")
ax.set_title("Cached hotspot records per month (detections, not fire events)")
ax.set_ylabel("records")

Nulls are split by origin against each sensor's column contract. A column is
`schema_absent` only when *another* sensor family or processing mode is the one that
sends it (`MODIS` has no `bright_ti4_k`, `VIIRS` has no `bright_t31_k`, only SP rows
carry `type`). Any other column that is empty for a whole sensor — `frp_mw` above
all — is `all_null_unexplained`: its origin is unknown and it may be a failed
collection. A column empty for only *some* rows is `partial_null`.

None of these may be summed as zero: an uncollected FRP counted as 0 would invent a
clean hour.

In [ ]:
groups = report["hotspot_audit"]["null_origin"]["groups"]
pd.DataFrame(
    [
        {
            "source_sensor": name,
            "rows": group["rows"],
            "schema_absent": ", ".join(group["schema_absent"]) or "-",
            "partial_null": ", ".join(f"{c}={n}" for c, n in group["partial_null"].items()) or "-",
            "all_null_unexplained": ", ".join(group["all_null_unexplained"]) or "-",
        }
        for name, group in groups.items()
    ]
).set_index("source_sensor")

## 7 · Value checks

Impossible values abort the run, so that list is empty by construction. Extreme-but-possible values
are reported here and left unchanged.

In [ ]:
extremes = report["value_checks"]["extreme_reported_not_removed"]
print(report["value_checks"]["policy"])
rows = [
    {"source": source, "column": column, **stats}
    for source, per_source in extremes.items()
    for column, stats in per_source.items()
]
if rows:
    pd.DataFrame(rows)
else:
    print("No value outside the documented extreme envelope.")

## 8 · Limits carried by this snapshot

In [ ]:
for line in report["limitations"]:
    print("-", line)

## 9 · Save the interim outputs

In [ ]:
saved = save_prepared(hourly, report, DATA_DIR)
print("table  :", saved["interim_path"])
print("sha256 :", saved["hourly_sha256"])
print("report :", Path(saved["interim_path"]).parent / "audit_report.json")

Next: CP2 (`notebooks/03_feature_engineering.ipynb`) reads these two files from disk — not from this
kernel — re-checks `hourly_sha256` against the audit report, and builds
`ml/data/processed/model_table.parquet`.